# Step 10.2 — Fixed-Vector Bottleneck：为什么“整段 Source → 一个 Final State”会变难

Step 10.1 建立了经典 recurrent encoder–decoder Seq2Seq：encoder 先读完整个 source sequence，再把 final encoder state `s=(B,H)` 作为 decoder initial hidden state。

这一节只分析这个 bridge 的局限：

```text
source hidden-state trajectory
(B,T_src,H)
        ↓ take final state only
context state s
(B,H)
        ↓
decoder
```

核心问题不是“RNN 完全不能表示长历史”，而是：**经典 Seq2Seq 让 decoder 只能通过一个固定宽度 final state 接收整个 source 的信息。**

本节不进入 Step 10.3，不介绍 attention score、attention weights 或 weighted sum。


## 1. Bottleneck 发生在哪里？不是 Encoder 没有中间 States，而是 Decoder 看不到它们

Step 10.1 的 encoder 实际会产生整条 state trajectory：

$$
H^{enc}
\in
\mathbb R^{B\times T_{src}\times H}.
$$

其中每个 source 位置都有自己的 encoder hidden state：

$$
h_0^{enc},h_1^{enc},\ldots,h_{T_{src}-1}^{enc}.
$$

但是经典 fixed-vector Seq2Seq bridge 只取：

$$
\boxed{
s=h_{T_{src}-1}^{enc}
}
$$

然后：

$$
h_{-1}^{dec}=s.
$$

所以真正的 bottleneck 是 **interface bottleneck（接口瓶颈）**：

```text
encoder 内部曾经有 T_src 个 states
↓
decoder 不直接接收这些 states
↓
只接收一个 final state s
```

这点很重要：不能误写成“encoder 只产生一个 state”。Encoder 有整条 trajectory；只是经典 decoder bridge 只暴露 final state。


In [ ]:
import mlx.core as mx
import mlx.nn as nn

B = 2
C_src = 6
H = 8

encoder = nn.GRU(input_size=C_src, hidden_size=H)

for T_src in [4, 16, 64]:
    source_x = mx.zeros((B, T_src, C_src))
    encoder_states = encoder(source_x)       # (B,T_src,H)
    context_state = encoder_states[:, -1, :] # (B,H)

    print(
        f'T_src={T_src:2d} | '
        f'encoder_states={encoder_states.shape} | '
        f'context_state={context_state.shape}'
    )


## 2. Fixed-Vector 是什么意思？T_src 变长，但 Bridge Width 仍然是 H

假设：

```text
H = 8
```

那么无论：

```text
T_src = 4
T_src = 16
T_src = 64
```

最终传给 decoder 的 context state 始终是：

$$
s\in\mathbb R^{B\times 8}.
$$

这就是 **fixed-vector bottleneck（固定向量瓶颈）** 中的 “fixed-vector”：

> source sequence length 可以变化，但 encoder→decoder 接口的表示宽度固定为 H。

随着 source 变长，模型必须让同一个 H-dimensional final state 同时保留越来越多、越来越分散的任务相关信息。

### 但不要做错误的维度证明

不能因为 `T_src > H` 就断言“数学上绝对不可能编码 source”。连续实数理论上可以承载复杂编码。

现实困难来自：

```text
有限数值精度
有限 H 与参数量
可训练的连续 recurrent update
优化与梯度传播困难
噪声
decoder 需要从 s 中可用地恢复任务相关细节
```

因此这里说的是 **practical representation bottleneck（实际表示瓶颈）**，不是纯粹的维度计数不可能性。


## 3. 早期 Source 信息到晚期 Target，还要走一条很长的 Recurrent Path

再引入两个索引：

```text
j = source position, 0 ... T_src-1
u = decoder/target position, 0 ... T_tgt-1
```

考虑一个很早的 source token `x_j` 要影响 decoder 的第 `u` 个 hidden state。

在经典 final-state bridge 中，它必须先沿 encoder recurrence 走到：

$$
h_{T_{src}-1}^{enc}=s.
$$

source-side recurrent transitions 数量为：

$$
T_{src}-1-j.
$$

然后 `s=h_{-1}^{dec}` 还要沿 decoder recurrence 传播到 `h_u^{dec}`，需要：

$$
u+1
$$

个 decoder recurrent transitions。

所以如果只数 recurrent transitions，这条 path 长度是：

$$
\boxed{
(T_{src}-1-j)+(u+1)
=
T_{src}+u-j
}
$$

例如 `T_src=20`，最早 source position `j=0` 要影响 decoder 的 `u=9`：

$$
20+9-0=29
$$

个 recurrent transitions。

门控可以改善这些 transitions 上的信息与 gradient flow，但 classic Seq2Seq 并没有为早期 source position 建立一条到 decoder 的短直连路径。


In [ ]:
def recurrent_path_length(T_src, source_index, decoder_index):
    encoder_steps = T_src - 1 - source_index
    decoder_steps = decoder_index + 1
    return encoder_steps + decoder_steps

T_src = 20

for source_index in [0, 10, 19]:
    for decoder_index in [0, 9]:
        length = recurrent_path_length(
            T_src, source_index, decoder_index
        )
        print(
            f'source j={source_index:2d}, decoder u={decoder_index:2d} '
            f'-> recurrent path length={length}'
        )


## 4. 为什么 Source 越长，这个结构越容易变难？

这里有两个不同但叠加的问题。

### A. Representation Pressure（表示压力）

decoder 只拿到一个：

$$
s=(B,H).
$$

如果 source 中有多个相隔很远但都与输出有关的细节，encoder 必须让这些信息经过 recurrent updates 后仍然在 final state 中可用。

例如翻译长句时，decoder 后面某一步可能需要 source 前部的实体、数值、语法关系或上下文细节。经典 bridge 没有让 decoder 直接回到那个 source position；它只能依赖这些细节是否成功保存在 `s` 中。

### B. Long Credit Path（长信用分配路径）

训练时，如果 decoder 后期某个 prediction 出错，loss 对早期 source processing 的 credit 也要沿：

```text
decoder recurrence
↓
final encoder state s
↓
encoder recurrence
↓
early source states
```

传播。

所以 Step 9 的 long-range credit assignment 问题并没有消失，只是现在跨越了 encoder 与 decoder 两段 recurrent chain。

### Teacher Forcing 不能解决这个 Bottleneck

Teacher forcing 让 decoder 在训练时看到正确 previous target token，可以让 decoder-side training 更直接。

但它没有改变：

```text
source information
↓
final encoder state s
↓
decoder
```

这个接口结构。

因此 teacher forcing 与 fixed-vector bottleneck 是两个不同问题。


## 5. 增大 H、用 LSTM/GRU 能缓解，但为什么仍不改变问题本身？

可以尝试：

```text
更大的 H
更强的 LSTM / GRU
更好的训练
更短或更简单的 source
```

这些都可能改善实际效果。

例如 H 从 128 增加到 512，final state 有更大的 representation capacity。

但经典架构的接口仍然是：

$$
H^{enc}\in\mathbb R^{B\times T_{src}\times H}
\quad\longrightarrow\quad
s\in\mathbb R^{B\times H}
\quad\longrightarrow\quad
decoder.
$$

也就是说：

```text
所有 source positions
↓
必须先汇总进 final state
↓
decoder 才能使用
```

所以这些措施主要是在**提高 bottleneck 的容量或可训练性**，而不是改变 decoder 访问 source information 的方式。

这就是 Step 10.2 最重要的历史问题：

> 如果 encoder 明明已经产生了每个 source position 的 hidden states，为什么 decoder 只能依赖最后一个？

这个问题会推动下一小节的历史发展，但本轮不回答“如何让 decoder 选择 source states”，也不介绍任何 Attention 计算。

### Step 10.2 材料边界

- fixed-vector bottleneck 指 classic recurrent Seq2Seq 只把 final encoder state `(B,H)` 暴露给 decoder。
- encoder 本身有完整 `(B,T_src,H)` state trajectory；不能误写成 encoder 只有一个 state。
- `T_src` 增长时 context width H 不增长，形成实际 representation pressure，但不是简单维度计数证明的数学不可能性。
- early source → late target 的影响路径同时穿过 encoder 与 decoder recurrent transitions。
- teacher forcing 不改变 encoder→decoder 的 fixed-vector interface。
- 增大 H 或使用更强 gated RNN 可以缓解容量/优化问题，但不改变“decoder 只拿 final state”的接口。
- 本轮只准备 Step 10.2；不进入 Step 10.3，不介绍 Attention 机制。
